# Analyse d'un scénario de réseau LoRaWAN

Ce notebook exécute le scénario `examples/quartier_sf9.yaml` (500 compteurs autour d'une passerelle),
présente les indicateurs et étudie la sensibilité au shadowing.

Le calcul est fait par le cœur natif C++ ; le notebook n'assemble que les résultats.
Prérequis : `build/liblsim_core.so` compilé (voir le README) et `pip install -e python[report]`
ou exécution depuis le dépôt (le notebook ajoute `python/` au chemin).

In [1]:
import sys
from dataclasses import replace
from pathlib import Path
from collections import defaultdict

sys.path.insert(0, str(Path.cwd().parent / "python"))

from lsim.export import run_scenario
from lsim.scenario import load_scenario
from lsim.simulator import simulate_uplinks

ROOT = Path.cwd().parent
scenario = load_scenario(ROOT / "examples" / "quartier_sf9.yaml")
result = run_scenario(scenario)
print(f"scénario : {result.scenario_name}")
print(f"empreinte : {result.scenario_sha256[:16]}…")

scénario : Quartier 500 compteurs SF9
empreinte : 07a43b99824497c7…


## Indicateurs globaux

In [2]:
t = result.totals
g = result.gateway
print(f"émises : {t.sent}   reçues : {t.received}   PDR : {result.pdr * 100:.2f} %")
print(f"pertes sensibilité : {t.lost_sensitivity}   pertes collision : {t.lost_collision}")
print(f"passerelle : occupée {g.busy_ns / 1e9:.1f} s sur {g.observation_ns / 1e9:.1f} s "
      f"(taux {g.utilization * 100:.2f} %), pic {g.peak_concurrent} trames simultanées")

émises : 5000   reçues : 4210   PDR : 84.20 %
pertes sensibilité : 0   pertes collision : 790
passerelle : occupée 880.0 s sur 8996.0 s (taux 9.78 %), pic 3 trames simultanées


## PDR par distance
Le PDR est regroupé par tranches de 500 m autour de la passerelle.

In [3]:
bins = defaultdict(lambda: [0, 0])  # tranche -> [émises, reçues]
for r in result.rows:
    d = (r.x_m ** 2 + r.y_m ** 2) ** 0.5
    b = int(d // 500) * 500
    bins[b][0] += r.sent
    bins[b][1] += r.received
print(f"{'distance (m)':>14} {'émises':>8} {'reçues':>8} {'PDR':>8}")
for b in sorted(bins):
    sent, recv = bins[b]
    pdr = recv / sent * 100 if sent else 0.0
    print(f"{b:>6}-{b + 500:<7} {sent:>8} {recv:>8} {pdr:>7.1f} %")

  distance (m)   émises   reçues      PDR
     0-500          170      170   100.0 %
   500-1000         380      380   100.0 %
  1000-1500         640      530    82.8 %
  1500-2000         950      790    83.2 %
  2000-2500        1270     1050    82.7 %
  2500-3000        1590     1290    81.1 %


## Sensibilité au shadowing

Le shadowing log-normal est tiré par dispositif (graine fixe). On fait varier son écart-type
et on compare le PDR global. Les dispositifs et la graine restent identiques entre les cas.

In [4]:
devices, _groups, _indices = scenario.deploy()
base = scenario.propagation()
print(f"{'sigma (dB)':>10} {'PDR':>8} {'pertes sensibilité':>20}")
for sigma in (0.0, 4.0, 8.0):
    prop = replace(base, shadowing_sigma_db=sigma, shadowing_seed=42)
    res = simulate_uplinks(devices, scenario.gateway(), prop, scenario.environment)
    pdr = res.totals.received / res.totals.sent * 100
    print(f"{sigma:>10.1f} {pdr:>7.2f} % {res.totals.lost_sensitivity:>20}")

sigma (dB)      PDR   pertes sensibilité
       0.0   84.20 %                    0
       4.0   85.60 %                    0
       8.0   87.00 %                    0


**Lecture.** Ici le PDR *augmente* avec le shadowing. Avec le modèle de collision de base
(capture à 6 dB), une dispersion des niveaux reçus favorise les signaux forts, qui survivent plus
souvent aux collisions. C'est un effet du modèle, à ne pas lire comme une prédiction terrain.
Le shadowing réel est à calibrer sur mesures.

## Notes de modélisation

- Modèle de collision de base : même SF, capture à 6 dB ; pas de réception multi-passerelle.
- Pas de dérive d'horloge, pas de downlink dans ce scénario.
- Le PDR ne tient pas compte des retransmissions (ni de l'ACK) : c'est le taux de première réception.
- Les valeurs de propagation sont des paramètres du scénario, pas des mesures : à calibrer
  (`python -m lsim calibrate`) avant tout usage contractuel.